# Reporte de esbeltez local y global desde el .s2k (sin resultados)

Sube el modelo `.s2k` (o `.txt`) **sin correr**. El notebook calcula la esbeltez local (NCh2369:2025 Tabla 9, con Ry·Fy) y global (KL/r, 8.8.4 para arriostramientos) de cada sección por grupo, y genera `reporte_esbeltez.xlsx` en `/kaggle/working`.

Parámetros editables en la celda 1: Ry, límite KL/r de columnas y vigas, Ca por defecto y Kx/Ky (por grupo, en la tabla `PARAM`).

In [ ]:
# ===== REPORTE DE ESBELTEZ LOCAL Y GLOBAL DESDE EL .s2k (sin resultados) =====
import math, re, glob, os, io, gzip, base64
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ---------- 1. CONFIGURACIÓN (editable) ----------
INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
MODEL_FILE = None              # None = detectar el .s2k (o .txt) en INPUT_DIR
E_MPA = 200000.0
RY = 1.5                        # Ry: confirmar en la NCh2369:2025
FY_OVERRIDE_KGF_CM2 = None      # None = usar el Fy del material del modelo
LIM_KLR_COLUMNAS_VIGAS = 200.0  # límite KL/r para columnas y vigas (cambiar si la norma lo indica)
CA_DEFAULT = 0.114              # Ca de la verificación del alma (cambiar por grupo si conoces la carga axial)
K_DEFAULT = 1.0

# ---------- 2. LECTURA DEL .s2k ----------
KV = re.compile(r'(\w+)=("[^"]*"|\S+)')
def find_model():
    if MODEL_FILE: return MODEL_FILE
    hits = [p for p in glob.glob(f'{INPUT_DIR}/**/*', recursive=True) if p.lower().endswith(('.s2k', '.txt'))]
    hits = [p for p in hits if 'TABLE:' in open(p, encoding='latin-1', errors='replace').read(4000)]
    assert len(hits) == 1, f'Se esperaba un .s2k/.txt en {INPUT_DIR}; encontrados: {hits}'
    return hits[0]

def logical_lines(path):
    buf = ''
    with open(path, encoding='latin-1', errors='replace') as f:
        for line in f:
            s = line.rstrip('\n')
            if s.rstrip().endswith('_'):
                buf += s.rstrip()[:-1] + ' '; continue
            yield buf + s; buf = ''

WANT = {'PROGRAM CONTROL', 'CONNECTIVITY - FRAME', 'FRAME SECTION ASSIGNMENTS', 'GROUPS 2 - ASSIGNMENTS',
        'FRAME SECTION PROPERTIES 01 - GENERAL', 'MATERIAL PROPERTIES 03A - STEEL DATA', 'MATERIAL PROPERTIES 01 - GENERAL'}
def read_model(path):
    out = {k: [] for k in WANT}; cur = None
    for ln in logical_lines(path):
        s = ln.strip()
        if s.startswith('TABLE:'):
            m = re.match(r'TABLE:\s+"(.+?)"', s); cur = m.group(1) if m else None; continue
        if cur in WANT and '=' in s:
            out[cur].append({k: v.strip('"') for k, v in KV.findall(s)})
    return out

model_path = find_model()
raw = read_model(model_path)
units = raw['PROGRAM CONTROL'][0].get('CurrUnits', '') if raw['PROGRAM CONTROL'] else ''
assert [p.strip().lower() for p in units.split(',')][:2] == ['kgf', 'm'], f'Unidades: {units}. Se requiere kgf-m.'
print('Modelo:', os.path.basename(model_path), '| unidades:', units)

# material y Fy
mat = {r['Material']: r for r in raw['MATERIAL PROPERTIES 03A - STEEL DATA']}
fy_modelo_kgf_cm2 = {k: float(v['Fy']) / 1e4 for k, v in mat.items() if 'Fy' in v}   # kgf/m² -> kgf/cm²
print('Fy del modelo (kgf/cm²):', {k: round(v, 1) for k, v in fy_modelo_kgf_cm2.items()})

# secciones
secciones = {}
for r in raw['FRAME SECTION PROPERTIES 01 - GENERAL']:
    secciones[r['SectionName']] = r

# barras, grupos, longitudes
conn = {r['Frame']: float(r['Length']) for r in raw['CONNECTIVITY - FRAME']}
sec_asig = {r['Frame']: r['AnalSect'] for r in raw['FRAME SECTION ASSIGNMENTS']}
grupos = {}
for r in raw['GROUPS 2 - ASSIGNMENTS']:
    if r.get('ObjectType') == 'Frame':
        grupos.setdefault(r['GroupName'], []).append(r['ObjectLabel'])
grupos = {g: fr for g, fr in grupos.items() if g != 'All'}
print('Grupos:', {g: len(f) for g, f in grupos.items()})

# ---------- 3. PARÁMETROS POR GRUPO (editable) ----------
def rol_por_nombre(g):
    n = g.upper()
    if 'ARR' in n or 'DIAG' in n or 'PUNT' in n: return 'Arriostramiento'
    if 'VIG' in n: return 'Viga'
    return 'Columna'
PARAM = pd.DataFrame([{'grupo': g, 'rol': rol_por_nombre(g), 'Kx': K_DEFAULT, 'Ky': K_DEFAULT,
                       'Ca': CA_DEFAULT, 'lim_KLr': LIM_KLR_COLUMNAS_VIGAS, 'Fy_kgf_cm2': None, 'Ry': RY}
                      for g in grupos])
# Ejemplo para ajustar: PARAM.loc[PARAM.grupo == 'COLUMNAS', 'Ca'] = 0.05
display_df = PARAM
print(PARAM.to_string(index=False))

# ---------- 4. GEOMETRÍA Y PROPIEDADES POR SECCIÓN ----------
def inercia_I(H, B, tf, tw):
    """Ix, Iy (m⁴) de una sección I doblemente simétrica."""
    Ix = B * H ** 3 / 12 - (B - tw) * (H - 2 * tf) ** 3 / 12
    Iy = 2 * tf * B ** 3 / 12 + (H - 2 * tf) * tw ** 3 / 12
    return Ix, Iy

import gzip as _gz, base64 as _b64
DB_GZ_B64 = (
    "..."
)
_DBLIST = __import__('pandas').read_csv(__import__('io').StringIO(_gz.decompress(_b64.b64decode(DB_GZ_B64)).decode('utf-8')))
_ANG = _DBLIST[_DBLIST.tipo == 'L (ángulo)']

def rmin_angulo(leg_mm, t_mm):
    """r_min (cm) del ángulo en la base de perfiles: mismo lado y espesor (±0,5 mm)."""
    c = _ANG[((_ANG.H_mm - leg_mm).abs() < 0.5) & ((_ANG.tw_mm - t_mm).abs() < 0.5)]
    c = c[c.r_min_cm.notna()]
    return float(c.r_min_cm.iloc[0]) if len(c) else None


def geom(nombre):
    s = secciones.get(nombre)
    if s is None: return None
    shape = s.get('Shape', '')
    A = float(s['Area'])
    if shape.startswith('I/'):
        H, B, tf, tw = float(s['t3']), float(s['t2']), float(s['tf']), float(s['tw'])
        Ix = float(s['I33']) if s.get('I33') else inercia_I(H, B, tf, tw)[0]
        Iy = float(s['I22']) if s.get('I22') else inercia_I(H, B, tf, tw)[1]
        return dict(tipo='I', H=H * 1000, B=B * 1000, tf=tf * 1000, tw=tw * 1000, A=A * 1e4,
                    rx=math.sqrt(Ix / A) * 100, ry=math.sqrt(Iy / A) * 100, nota='')
    if shape == 'Angle':
        leg, t = float(s['t3']) * 1000, float(s['tf']) * 1000
        I = float(s['I33'])
        rmin = rmin_angulo(leg, t)
        if rmin is None:
            rmin = math.sqrt(I / A) * 100 * 0.6
            nota = 'r_min aproximado (0,6·r_x): no está en la base'
        else:
            nota = 'r_min de la base de perfiles'
        return dict(tipo='L', H=leg, B=leg, tf=t, tw=t, A=A * 1e4, rx=math.sqrt(I / A) * 100, ry=rmin, nota=nota)
    return None

# ---------- 5. CÁLCULO ----------
def lim_local(g, Ca, Ry):
    rf = Ry * 1  # placeholder
def cumple_local(geo, rol, Ca, RyFy):
    filas = []
    k_ala = 0.40 * math.sqrt(E_MPA / RyFy)
    if geo['tipo'] == 'I':
        filas.append(('Ala (b/t)', (geo['B'] / 2) / geo['tf'], k_ala))
        hw = geo['H'] - 2 * geo['tf']
        if rol == 'Arriostramiento':
            lim = 1.57 * math.sqrt(E_MPA / RyFy)
        elif Ca <= 0.114:
            lim = 3.96 * math.sqrt(E_MPA / RyFy) * (1 - 3.04 * Ca)
        else:
            lim = 1.57 * math.sqrt(E_MPA / RyFy)
        filas.append(('Alma (h/t)', hw / geo['tw'], lim))
    elif geo['tipo'] == 'L':
        filas.append(('Pata (b/t)', geo['H'] / geo['tf'], k_ala))
    return [(n, round(l, 2), round(lm, 2), l <= lm) for n, l, lm in filas]

RESUMEN, DETALLE = [], []
for _, p in PARAM.iterrows():
    g = p['grupo']
    Fy = p['Fy_kgf_cm2'] or FY_OVERRIDE_KGF_CM2 or next(iter(fy_modelo_kgf_cm2.values()))
    Fy_MPA = Fy * 0.0980665
    RyFy = p['Ry'] * Fy_MPA
    lim_glob = (1.5 * math.pi * math.sqrt(E_MPA / Fy_MPA)) if p['rol'] == 'Arriostramiento' else p['lim_KLr']
    secs = {}
    for fr in grupos[g]:
        sn = sec_asig.get(fr); secs.setdefault(sn, []).append(fr)
    for sn, frames in secs.items():
        geo = geom(sn)
        if geo is None:
            RESUMEN.append(dict(grupo=g, rol=p['rol'], seccion=sn, nota=f'Tipo no verificado ({secciones.get(sn, {}).get("Shape", "sin sección")})'))
            continue
        Lmax = max(conn[f] for f in frames)
        lam_x = max(p['Kx'] * conn[f] * 100 / geo['rx'] for f in frames)
        lam_y = max(p['Ky'] * conn[f] * 100 / geo['ry'] for f in frames)
        lam = max(lam_x, lam_y)
        locales = cumple_local(geo, p['rol'], p['Ca'], RyFy)
        ok_loc = all(c for *_, c in locales)
        ok_glob = lam <= lim_glob
        RESUMEN.append(dict(grupo=g, rol=p['rol'], seccion=sn, tipo=geo['tipo'], n_barras=len(frames),
            barras=', '.join(frames), L_max_m=round(Lmax, 3), Kx=p['Kx'], Ky=p['Ky'],
            rx_cm=round(geo['rx'], 2), ry_cm=round(geo['ry'], 2), lambda_x=round(lam_x, 1), lambda_y=round(lam_y, 1),
            lambda_max=round(lam, 1), limite_global=round(lim_glob, 1), global_cumple=ok_glob,
            **{f'{n} λ': l for n, l, _, _ in locales}, **{f'{n} λmd': lm for n, _, lm, _ in locales},
            local_cumple=ok_loc, Ca_usado=p['Ca'], cumple_todo=ok_loc and ok_glob,
            nota=' | '.join(x for x in [geo['nota'], 'Ca supuesto: revisar' if geo['tipo'] == 'I' else ''] if x)))
        for f in frames:
            DETALLE.append(dict(grupo=g, barra=f, seccion=sn, L_m=round(conn[f], 3),
                                lambda_x=round(p['Kx'] * conn[f] * 100 / geo['rx'], 1),
                                lambda_y=round(p['Ky'] * conn[f] * 100 / geo['ry'], 1),
                                limite_global=round(lim_glob, 1)))
RES = pd.DataFrame(RESUMEN)
DET = pd.DataFrame(DETALLE)
print(RES[['grupo', 'seccion', 'lambda_max', 'limite_global', 'global_cumple', 'local_cumple', 'cumple_todo']].to_string(index=False))

# ---------- 6. EXCEL ----------
os.makedirs(OUTPUT_DIR, exist_ok=True)
xlsx = os.path.join(OUTPUT_DIR, 'reporte_esbeltez.xlsx')
wb = Workbook(); wb.remove(wb.active)
HDR = PatternFill('solid', fgColor='1F4E78'); OK = PatternFill('solid', fgColor='C6EFCE'); BAD = PatternFill('solid', fgColor='FFC7CE')
def escribir(ws, df, titulo):
    ws.append([titulo]); ws['A1'].font = Font(bold=True, size=13)
    ws.append([])
    if df.empty:
        ws.append(['Sin datos']); return
    ws.append(list(df.columns))
    for c in ws[3]: c.font = Font(bold=True, color='FFFFFF'); c.fill = HDR; c.alignment = Alignment(wrap_text=True, vertical='center')
    for _, r in df.iterrows(): ws.append([None if (isinstance(v, float) and math.isnan(v)) else v for v in r.tolist()])
    for col in range(1, len(df.columns) + 1):
        ws.column_dimensions[get_column_letter(col)].width = min(max(12, len(str(df.columns[col - 1])) + 2), 28)
    for row in ws.iter_rows(min_row=4, max_row=ws.max_row):
        for c in row:
            if isinstance(c.value, bool):
                c.fill = OK if c.value else BAD
    ws.freeze_panes = 'A4'
ws = wb.create_sheet('Resumen')
ws.append(['Reporte de esbeltez local y global (NCh2369:2025 Tabla 9 y 8.8.4)'])
ws['A1'].font = Font(bold=True, size=14)
ws.append([f'Modelo: {os.path.basename(model_path)}', '', f'Ry = {RY}', f'E = {E_MPA:.0f} MPa'])
ws.append(['Notas: λ local con Ry·Fy; ángulos y Ca según la hoja de cada grupo; r_min de ángulos aproximado (confirmar).'])
ws.append([])
resumen_cols = ['grupo', 'seccion', 'tipo', 'n_barras', 'L_max_m', 'lambda_max', 'limite_global', 'global_cumple', 'local_cumple', 'cumple_todo', 'nota']
resumen_df = RES.reindex(columns=resumen_cols)
ws.append(list(resumen_df.columns))
for c in ws[ws.max_row]: c.font = Font(bold=True, color='FFFFFF'); c.fill = HDR
for _, r in resumen_df.iterrows(): ws.append([None if (isinstance(v, float) and math.isnan(v)) else v for v in r.tolist()])
for row in ws.iter_rows(min_row=ws.max_row - len(resumen_df) + 1, max_row=ws.max_row):
    for c in row:
        if isinstance(c.value, bool): c.fill = OK if c.value else BAD
for col in range(1, len(resumen_cols) + 1): ws.column_dimensions[get_column_letter(col)].width = 18
for g in grupos:
    sub = RES[RES.grupo == g].drop(columns=['grupo'], errors='ignore')
    escribir(wb.create_sheet(g[:28]), sub, f'Grupo: {g} ({PARAM.loc[PARAM.grupo == g, "rol"].iloc[0]})')
escribir(wb.create_sheet('Detalle por barra'), DET, 'Detalle por barra')
wb.save(xlsx)
print('Reporte guardado en:', xlsx)
